### Notebook 26

# **Regions of Interest Statistical Tests**

## **1. Machine Learning**

We perform **Wilcoxon signed-rank tests** on the results of the machine learning models for the classification and regression tasks. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import scipy
import pingouin
import os

In [2]:
# Define the preprocessed data path. 
preprocessed_data_path = '../neuropolis-x1_preprocessed_data/'

# Define the results path. 
results_path = '../neuropolis-x1_results/'

In [3]:
# Load the targets. 
with open(preprocessed_data_path + 'dict_targets.p', 'rb') as file:
    dict_targets = pickle.load(file)
with open(preprocessed_data_path + 'classification/dict_targets_classification_sequence.p', 'rb') as file:
    dict_targets_class = pickle.load(file)
with open(preprocessed_data_path + 'regression/dict_targets_regression_sequence.p', 'rb') as file:
    dict_targets_reg = pickle.load(file)
with open(preprocessed_data_path + 'classification/dict_targets_classification_basis.p', 'rb') as file:
    dict_targets_foundation_models = pickle.load(file)

# Define the list of subjects, removing sub-xp102 who has a missing condition. 
subjects = ['sub-xp1' + str(x).zfill(2) for x in range(1, 11)]
subjects.remove('sub-xp102')
subject = subjects[0]

# Retrieve and display the brain region names and the number of brain regions. 
brain_regions = list(dict_targets[subject]['eegfmriNF'].keys())
print(brain_regions)
print(len(brain_regions), 'brain regions')

['Background', 'Frontal Pole', 'Insular Cortex', 'Superior Frontal Gyrus', 'Middle Frontal Gyrus', 'Inferior Frontal Gyrus, pars triangularis', 'Inferior Frontal Gyrus, pars opercularis', 'Precentral Gyrus', 'Temporal Pole', 'Superior Temporal Gyrus, anterior division', 'Superior Temporal Gyrus, posterior division', 'Middle Temporal Gyrus, anterior division', 'Middle Temporal Gyrus, posterior division', 'Middle Temporal Gyrus, temporooccipital part', 'Inferior Temporal Gyrus, anterior division', 'Inferior Temporal Gyrus, posterior division', 'Inferior Temporal Gyrus, temporooccipital part', 'Postcentral Gyrus', 'Superior Parietal Lobule', 'Supramarginal Gyrus, anterior division', 'Supramarginal Gyrus, posterior division', 'Angular Gyrus', 'Lateral Occipital Cortex, superior division', 'Lateral Occipital Cortex, inferior division', 'Intracalcarine Cortex', 'Frontal Medial Cortex', 'Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)', 'Subcallosal Cortex', 'Paracingulate

In [4]:
# Define a function to create the DataFrame for the statistical tests. 
def create_df(dict_targets, dict_predictions, brain_regions, score_type, model_type, test_set):

    # Define Pandas DataFrames to store the results. 
    df = pd.DataFrame(columns = ['subject', 'brain_region', 'model_' + score_type, 'baseline_' + score_type])
    counter = 0

    # Iterate through all subjects. 
    for subject in subjects:

        # Iterate through all brain regions. 
        for brain_region_index in range(len(brain_regions)):

            # Fill the DataFrame with the subject and brain region. 
            df.loc[counter, 'subject'] = subject
            df.loc[counter, 'brain_region'] = brain_regions[brain_region_index]

            # Retrieve the targets and predictions values for the current subject and brain region. 
            targets_values = dict_targets[subject][test_set][:, brain_region_index]
            if model_type == 'machine_learning':
                predictions_values = dict_predictions[subject][test_set][:, brain_region_index]
            elif model_type == 'deep_learning':
                predictions_values = dict_predictions[subject][:, brain_region_index]

            # For the classification task, compute the accuracy. 
            if score_type == 'accuracy':
                model_score = np.mean(targets_values == predictions_values)
                baseline_score = np.mean(targets_values == 1)
                
            # For the regression task, compute the mean absolute error (MAE).
            elif score_type == 'MAE':
                model_score = np.mean(np.abs(targets_values - predictions_values))
                baseline_score = np.mean(np.abs(targets_values - np.mean(targets_values)))

            # Fill the DataFrame with the model score and baseline score. 
            df.loc[counter, 'model_' + score_type] = model_score
            df.loc[counter, 'baseline_' + score_type] = baseline_score

            # Increment. 
            counter += 1

    # Return the DataFrame. 
    return df

In [5]:
# Example: Load the machine learning results for the classification task. 
with open(results_path + 'classification/dict_predictions_lr.p', 'rb') as file:
    dict_predictions_lr_class = pickle.load(file)
    
# Example: Create and display the DataFrame for the logistic regression model. 
df_lr_class = create_df(dict_targets_class, dict_predictions_lr_class, brain_regions, 'accuracy', 'machine_learning', 'fmriNF')
df_lr_class

,subject,brain_region,model_accuracy,baseline_accuracy
0,sub-xp101,Background,0.448454,0.5
1,sub-xp101,Frontal Pole,0.42268,0.561856
2,sub-xp101,Insular Cortex,0.525773,0.489691
3,sub-xp101,Superior Frontal Gyrus,0.494845,0.505155
4,sub-xp101,Middle Frontal Gyrus,0.474227,0.530928
...,...,...,...,...
436,sub-xp110,Planum Polare,0.479381,0.494845
437,sub-xp110,Heschl's Gyrus (includes H1 and H2),0.5,0.489691
438,sub-xp110,Planum Temporale,0.525773,0.489691
439,sub-xp110,Supracalcarine Cortex,0.536082,0.479381


In [6]:
# Example: Retrieve only the rows corresponding to the region 'Frontal Pole'. 
df_lr_class_frontal_pole = df_lr_class[df_lr_class['brain_region'] == 'Frontal Pole']
df_lr_class_frontal_pole

,subject,brain_region,model_accuracy,baseline_accuracy
1,sub-xp101,Frontal Pole,0.42268,0.561856
50,sub-xp103,Frontal Pole,0.530928,0.520619
99,sub-xp104,Frontal Pole,0.5,0.489691
148,sub-xp105,Frontal Pole,0.515464,0.5
197,sub-xp106,Frontal Pole,0.597938,0.551546
246,sub-xp107,Frontal Pole,0.592784,0.515464
295,sub-xp108,Frontal Pole,0.587629,0.484536
344,sub-xp109,Frontal Pole,0.541237,0.530928
393,sub-xp110,Frontal Pole,0.572165,0.530928


In [7]:
# Define a function to perform a Wilcoxon signed-rank test. 
def wilcoxon_test(df, score_type, brain_regions):

    # Create empty DataFrames to store the statistics. 
    statistics_scipy = pd.DataFrame()
    statistics_pingouin = pd.DataFrame()

    # Iterate through all brain regions. 
    for brain_region in brain_regions:

        # Retrieve only the rows of df where the column brain_region is equal to the current brain region. 
        roi_df = df[df['brain_region'] == brain_region]

        # Compute the mean and standard deviation of the model and the baseline. 
        mean_model_score = roi_df['model_' + score_type].mean()
        mean_baseline_score = roi_df['baseline_' + score_type].mean()
        std_model_score = roi_df['model_' + score_type].std()
        std_baseline_score = roi_df['baseline_' + score_type].std()

        # Perform the Wilcoxon signed-rank test using SciPy. 
        x = roi_df['model_' + score_type].astype(float).values
        y = roi_df['baseline_' + score_type].astype(float).values
        if score_type == 'accuracy':
            wilcoxon_stat, p_value = scipy.stats.wilcoxon(x, y, alternative = 'greater')
        elif score_type == 'MAE':
            wilcoxon_stat, p_value = scipy.stats.wilcoxon(x, y, alternative = 'less')

        # Compute the confidence interval of the mean. 
        res = scipy.stats.bootstrap((roi_df['model_' + score_type],), np.mean, confidence_level = 0.95)
        ci_low, ci_high = res.confidence_interval

        # Store the statistics in a DataFrame. 
        roi_statistics_scipy = pd.DataFrame([{
            'brain_region': brain_region, 
            'N': len(roi_df),
            'mean_model_' + score_type: mean_model_score,
            'mean_baseline_' + score_type: mean_baseline_score,
            'std_model_' + score_type: std_model_score,
            'std_baseline_' + score_type: std_baseline_score,
            'wilcoxon_stat': wilcoxon_stat,
            'p_value': p_value,
            'ci_low': ci_low, 
            'ci_high': ci_high
        }])

        # Perform the same test using Pingouin to obtain the RBC (Rank-Biserial Correlation) and CLES (Common Language Effect Size). 
        if score_type == 'accuracy':
            roi_statistics_pingouin = pingouin.wilcoxon(x, y, alternative = 'greater')
        elif score_type == 'MAE':
            roi_statistics_pingouin = pingouin.wilcoxon(y, x, alternative = 'greater') # Invert x and y for MAE to get the correct effect size direction. 
        
        # Add the brain region name to the Pingouin statistics DataFrame. 
        roi_statistics_pingouin.insert(0, 'brain_region', brain_region)
        
        # Concatenate the statistics to the main DataFrames. 
        statistics_scipy = pd.concat([statistics_scipy, roi_statistics_scipy], ignore_index = True)
        statistics_pingouin = pd.concat([statistics_pingouin, roi_statistics_pingouin], ignore_index = True)
    
    # Return the statistics. 
    return statistics_scipy, statistics_pingouin

In [8]:
# Perform the Wilcoxon signed-rank test. 
statistics_scipy_lr_class, statistics_pingouin_lr_class = wilcoxon_test(df_lr_class, 'accuracy', brain_regions)

In [9]:
# Display the SciPy statistics. 
statistics_scipy_lr_class

,brain_region,N,mean_model_accuracy,mean_baseline_accuracy,std_model_accuracy,std_baseline_accuracy,wilcoxon_stat,p_value,ci_low,ci_high
0,Background,9,0.537228,0.404353,0.046874,0.053637,44.0,0.003906,0.504573,0.562454
1,Frontal Pole,9,0.540092,0.520619,0.056453,0.026410,36.0,0.060547,0.495991,0.567583
2,Insular Cortex,9,0.532646,0.520619,0.028700,0.030277,31.0,0.171875,0.518900,0.556701
3,Superior Frontal Gyrus,9,0.529210,0.517182,0.030386,0.020457,28.5,0.251953,0.510882,0.548110
4,Middle Frontal Gyrus,9,0.549255,0.507446,0.042904,0.021581,39.0,0.027344,0.522080,0.575029
5,"Inferior Frontal Gyrus, pars triangularis",9,0.507446,0.509164,0.029399,0.012301,18.5,0.484375,0.488545,0.524628
6,"Inferior Frontal Gyrus, pars opercularis",9,0.523482,0.513173,0.039687,0.021270,26.0,0.355469,0.497136,0.546392
7,Precentral Gyrus,9,0.548110,0.518328,0.033899,0.025135,38.0,0.037109,0.522910,0.565865
8,Temporal Pole,9,0.541237,0.519473,0.027639,0.030796,37.5,0.039062,0.525773,0.560137
9,"Superior Temporal Gyrus, anterior division",9,0.499427,0.497136,0.025442,0.018062,23.5,0.466797,0.483963,0.515464


In [10]:
# Display the Pingouin statistics. 
statistics_pingouin_lr_class

,brain_region,W-val,alternative,p-val,RBC,CLES
0,Background,44.0,greater,0.003906,0.955556,0.981481
1,Frontal Pole,36.0,greater,0.060547,0.600000,0.679012
2,Insular Cortex,31.0,greater,0.171875,0.377778,0.561728
3,Superior Frontal Gyrus,28.5,greater,0.251953,0.266667,0.617284
4,Middle Frontal Gyrus,39.0,greater,0.027344,0.733333,0.814815
5,"Inferior Frontal Gyrus, pars triangularis",18.5,greater,0.484375,0.027778,0.518519
6,"Inferior Frontal Gyrus, pars opercularis",26.0,greater,0.355469,0.155556,0.641975
7,Precentral Gyrus,38.0,greater,0.037109,0.688889,0.814815
8,Temporal Pole,37.5,greater,0.039062,0.666667,0.691358
9,"Superior Temporal Gyrus, anterior division",23.5,greater,0.466797,0.044444,0.518519


In [11]:
# Define a function to create a summary DataFrame. 
def create_summary_df(model_names, score_type, brain_regions):
    
    # Create the summary DataFrame with a double index: model name and brain region. 
    df_summary = pd.DataFrame(index = pd.MultiIndex.from_product([model_names, brain_regions], names=['model', 'brain region']),
                              columns = ['N', 
                                         'mean model ' + score_type, 
                                         'mean baseline ' + score_type, 
                                         'STD model ' + score_type,
                                         'STD baseline ' + score_type,
                                         'Wilcoxon W statistic', 
                                         'p-value', 
                                         'CI (lower)',
                                         'CI (upper)', 
                                         'RBC', 
                                         'CLES'])
    
    return df_summary

In [12]:
# Display an example of empty summary DataFrame. 
create_summary_df(['Logistic Regression', 'K-Nearest Neighbors'], 'accuracy', brain_regions)

N  \
model               brain region                               
Logistic Regression Background                           NaN   
                    Frontal Pole                         NaN   
                    Insular Cortex                       NaN   
                    Superior Frontal Gyrus               NaN   
                    Middle Frontal Gyrus                 NaN   
...                                                      ...   
K-Nearest Neighbors Planum Polare                        NaN   
                    Heschl's Gyrus (includes H1 and H2)  NaN   
                    Planum Temporale                     NaN   
                    Supracalcarine Cortex                NaN   
                    Occipital Pole                       NaN   

                                                        mean model accuracy  \
model               brain region                                              
Logistic Regression Background                                          NaN   
                    Frontal Pole                                        NaN   
                    Insular Cortex                                      NaN   
                    Superior Frontal Gyrus                              NaN   
                    Middle Frontal Gyrus                                NaN   
...                                                                     ...   
K-Nearest Neighbors Planum Polare                                       NaN   
                    Heschl's Gyrus (includes H1 and H2)                 NaN   
                    Planum Temporale                                    NaN   
                    Supracalcarine Cortex                               NaN   
                    Occipital Pole                                      NaN   

                                                        mean baseline accuracy  \
model               brain region                                                 
Logistic Regression Background                                             NaN   
                    Frontal Pole                                           NaN   
                    Insular Cortex                                         NaN   
                    Superior Frontal Gyrus                                 NaN   
                    Middle Frontal Gyrus                                   NaN   
...                                                                        ...   
K-Nearest Neighbors Planum Polare                                          NaN   
                    Heschl's Gyrus (includes H1 and H2)                    NaN   
                    Planum Temporale                                       NaN   
                    Supracalcarine Cortex                                  NaN   
                    Occipital Pole                                         NaN   

                                                        STD model accuracy  \
model               brain region                                             
Logistic Regression Background                                         NaN   
                    Frontal Pole                                       NaN   
                    Insular Cortex                                     NaN   
                    Superior Frontal Gyrus                             NaN   
                    Middle Frontal Gyrus                               NaN   
...                                                                    ...   
K-Nearest Neighbors Planum Polare                                      NaN   
                    Heschl's Gyrus (includes H1 and H2)                NaN   
                    Planum Temporale                                   NaN   
                    Supracalcarine Cortex                              NaN   
                    Occipital Pole                                     NaN   

                                                        STD baseline accuracy  \
model               brain region   

In [13]:
# Define a function to store statistics in the summary DataFrame. 
def store_statistics_in_summary_df(df_summary, model_names, model_index, statistics_scipy, statistics_pingouin, score_type, brain_regions):

    # Iterate through all brain regions. 
    for brain_region in brain_regions:

        # Retrieve only the rows of statistics_scipy and statistics_pingouin where the column brain_region is equal to the current brain region. 
        roi_statistics_scipy = statistics_scipy[statistics_scipy['brain_region'] == brain_region]
        roi_statistics_pingouin = statistics_pingouin[statistics_pingouin['brain_region'] == brain_region]

        # Store the statistics in the summary DataFrame. 
        df_summary.loc[(model_names[model_index], brain_region), 'N'] = roi_statistics_scipy['N'].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'mean model ' + score_type] = roi_statistics_scipy['mean_model_' + score_type].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'mean baseline ' + score_type] = roi_statistics_scipy['mean_baseline_' + score_type].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'STD model ' + score_type] = roi_statistics_scipy['std_model_' + score_type].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'STD baseline ' + score_type] = roi_statistics_scipy['std_baseline_' + score_type].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'Wilcoxon W statistic'] = roi_statistics_scipy['wilcoxon_stat'].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'p-value'] = roi_statistics_scipy['p_value'].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'CI (lower)'] = roi_statistics_scipy['ci_low'].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'CI (upper)'] = roi_statistics_scipy['ci_high'].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'RBC'] = roi_statistics_pingouin['RBC'].values[0]
        df_summary.loc[(model_names[model_index], brain_region), 'CLES'] = roi_statistics_pingouin['CLES'].values[0]

        # Ensure the p-value column is numeric, then format it in scientific notation. 
        df_summary['p-value'] = pd.to_numeric(df_summary['p-value'], errors = 'coerce')
        df_summary['p-value'] = df_summary['p-value'].apply(lambda x: f'{x:.2e}')

    return df_summary

In [14]:
# Define a function to perform a Wilcoxon signed-rank test on a series of models. 
def run_wilcoxon_tests(model_names, dict_targets, list_dict_predictions, brain_regions, score_type, model_type, test_set):

    # Create a summary DataFrame to store the statistics for all models. 
    df_summary = create_summary_df(model_names, score_type, brain_regions)

    # Create a dictionary to store the metrics DataFrames for all models. 
    dict_df_metrics = {}

    # Create a dictionary to store all targets and predictions for all models. 
    dict_all_targets_predictions = {}

    # Iterate through all models. 
    for model_index in range(len(model_names)):

        # Create the DataFrame and perform the Wilcoxon signed-rank test. 
        dict_predictions = list_dict_predictions[model_index]
        df = create_df(dict_targets, dict_predictions, brain_regions, score_type, model_type, test_set)
        statistics_scipy, statistics_pingouin = wilcoxon_test(df, score_type, brain_regions)

        # Store the statistics in the summary DataFrame. 
        df_summary = store_statistics_in_summary_df(df_summary, model_names, model_index, statistics_scipy, statistics_pingouin, score_type, brain_regions)

        # Store the metrics DataFrame in the dictionary.
        dict_df_metrics[model_names[model_index]] = df
        
    # Return the summary DataFrame and the dictionary. 
    return df_summary, dict_df_metrics, dict_all_targets_predictions

### Classification

In [15]:
# Define a function to load the machine learning results for the classification task. 
def load_classification_ml_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'classification/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the machine learning results for the classification task. 
    with open(iteration_result_path + 'dict_predictions_lr.p', 'rb') as file:
        dict_predictions_lr_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_knn.p', 'rb') as file:
        dict_predictions_knn_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_dt.p', 'rb') as file:
        dict_predictions_dt_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_rf.p', 'rb') as file:
        dict_predictions_rf_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_svm.p', 'rb') as file:
        dict_predictions_svm_class = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_xgb.p', 'rb') as file:
        dict_predictions_xgb_class = pickle.load(file)

    return [dict_predictions_lr_class, 
            dict_predictions_knn_class, 
            dict_predictions_dt_class, 
            dict_predictions_rf_class, 
            dict_predictions_svm_class, 
            dict_predictions_xgb_class, 
            test_set]

In [16]:
# Define a function to run the tests for machine learning classification models.
def run_classification_ml_tests(cv_iteration, model_names):

    # Load the machine learning results for the classification task.
    dict_predictions_lr_class, dict_predictions_knn_class, dict_predictions_dt_class, dict_predictions_rf_class, dict_predictions_svm_class, dict_predictions_xgb_class, test_set = load_classification_ml_results(cv_iteration)

    predictions_ml_class = [dict_predictions_lr_class, 
                            dict_predictions_knn_class, 
                            dict_predictions_dt_class, 
                            dict_predictions_rf_class, 
                            dict_predictions_svm_class, 
                            dict_predictions_xgb_class]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_ml_class, dict_df_metrics_ml_class, _ = run_wilcoxon_tests(model_names, dict_targets_class, predictions_ml_class, brain_regions, 'accuracy', 'machine_learning', test_set)
    
    return df_summary_ml_class, dict_df_metrics_ml_class

In [17]:
# Define the machine learning models for classification. 
model_names = ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest', 'Support Vector Machine', 'XGBoost']

In [18]:
# Iteration 1: run the tests for machine learning classification models. 
cv_iteration = 1
df_summary_ml_class_iteration_1, dict_df_metrics_ml_class_iteration_1 = run_classification_ml_tests(cv_iteration, model_names)

In [19]:
# Iteration 2: run the tests for machine learning classification models. 
cv_iteration = 2
df_summary_ml_class_iteration_2, dict_df_metrics_ml_class_iteration_2 = run_classification_ml_tests(cv_iteration, model_names)

In [20]:
# Iteration 3: run the tests for machine learning classification models. 
cv_iteration = 3
df_summary_ml_class_iteration_3, dict_df_metrics_ml_class_iteration_3 = run_classification_ml_tests(cv_iteration, model_names)

In [21]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_ml_class_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_ml_class_pooled = create_summary_df(model_names, 'accuracy', brain_regions)

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_ml_class_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_ml_class_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_ml_class_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, 'accuracy', brain_regions)

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_ml_class_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_ml_class_pooled = store_statistics_in_summary_df(df_summary_ml_class_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, 'accuracy', brain_regions)

In [22]:
# Diplay the pooled summary DataFrame. 
df_summary_ml_class_pooled

N  \
model               brain region                              
Logistic Regression Background                           27   
                    Frontal Pole                         27   
                    Insular Cortex                       27   
                    Superior Frontal Gyrus               27   
                    Middle Frontal Gyrus                 27   
...                                                      ..   
XGBoost             Planum Polare                        27   
                    Heschl's Gyrus (includes H1 and H2)  27   
                    Planum Temporale                     27   
                    Supracalcarine Cortex                27   
                    Occipital Pole                       27   

                                                        mean model accuracy  \
model               brain region                                              
Logistic Regression Background                                     0.528255   
                    Frontal Pole                                   0.520809   
                    Insular Cortex                                   0.5168   
                    Superior Frontal Gyrus                         0.520237   
                    Middle Frontal Gyrus                           0.527873   
...                                                                     ...   
XGBoost             Planum Polare                                  0.512791   
                    Heschl's Gyrus (includes H1 and H2)            0.514128   
                    Planum Temporale                               0.509927   
                    Supracalcarine Cortex                            0.5042   
                    Occipital Pole                                 0.504773   

                                                        mean baseline accuracy  \
model               brain region                                                 
Logistic Regression Background                                         0.41428   
                    Frontal Pole                                      0.511264   
                    Insular Cortex                                    0.522719   
                    Superior Frontal Gyrus                            0.518328   
                    Middle Frontal Gyrus                              0.512218   
...                                                                        ...   
XGBoost             Planum Polare                                     0.506682   
                    Heschl's Gyrus (includes H1 and H2)               0.517373   
                    Planum Temporale                                  0.507064   
                    Supracalcarine Cortex                             0.494654   
                    Occipital Pole                                    0.507255   

                                                        STD model accuracy  \
model               brain region                                             
Logistic Regression Background                                      0.0389   
                    Frontal Pole                                  0.048492   
                    Insular Cortex                                0.033516   
                    Superior Frontal Gyrus                        0.036419   
                    Middle Frontal Gyrus                          0.037913   
...                                                                    ...   
XGBoost             Planum Polare                                 0.033963   
                    Heschl's Gyrus (includes H1 and H2)           0.039763   
                    Planum Temporale                              0.035509   
                    Supracalcarine Cortex                         0.044504   
                    Occipital Pole                                 0.03591   

                                                        STD baseline accuracy  \
model               brain region               

### Regression

In [23]:
# Define a function to load the machine learning results for the regression task. 
def load_regression_ml_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'regression/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the machine learning results for the regression task. 
    with open(iteration_result_path + 'dict_predictions_lr.p', 'rb') as file:
        dict_predictions_lr_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_ridge.p', 'rb') as file:
        dict_predictions_ridge_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_lasso.p', 'rb') as file:
        dict_predictions_lasso_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_knn.p', 'rb') as file:
        dict_predictions_knn_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_dt.p', 'rb') as file:
        dict_predictions_dt_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_rf.p', 'rb') as file:
        dict_predictions_rf_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_svm.p', 'rb') as file:
        dict_predictions_svm_reg = pickle.load(file)
    with open(iteration_result_path + 'dict_predictions_xgb.p', 'rb') as file:
        dict_predictions_xgb_reg = pickle.load(file)

    return [dict_predictions_lr_reg, 
            dict_predictions_ridge_reg, 
            dict_predictions_lasso_reg, 
            dict_predictions_knn_reg, 
            dict_predictions_dt_reg, 
            dict_predictions_rf_reg, 
            dict_predictions_svm_reg, 
            dict_predictions_xgb_reg, 
            test_set]

In [24]:
# Define a function to run the tests for machine learning regression models.
def run_regression_ml_tests(cv_iteration, model_names):
    
    # Load the machine learning results for the regression task.
    dict_predictions_lr_reg, dict_predictions_ridge_reg, dict_predictions_lasso_reg, dict_predictions_knn_reg, dict_predictions_dt_reg, dict_predictions_rf_reg, dict_predictions_svm_reg, dict_predictions_xgb_reg, test_set = load_regression_ml_results(cv_iteration)

    predictions_ml_reg = [dict_predictions_lr_reg,
                        dict_predictions_ridge_reg, 
                        dict_predictions_lasso_reg, 
                        dict_predictions_knn_reg, 
                        dict_predictions_dt_reg, 
                        dict_predictions_rf_reg, 
                        dict_predictions_svm_reg, 
                        dict_predictions_xgb_reg]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_ml_reg, dict_df_metrics_ml_reg, dict_all_targets_predictions_ml_reg = run_wilcoxon_tests(model_names, dict_targets_reg, predictions_ml_reg, brain_regions, 'MAE', 'machine_learning', test_set)
    
    return df_summary_ml_reg, dict_df_metrics_ml_reg, dict_all_targets_predictions_ml_reg

In [25]:
# Define the machine learning models for regression. 
model_names = ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest', 'Support Vector Machine', 'XGBoost']

In [26]:
# Iteration 1: run the tests for machine learning regression models. 
cv_iteration = 1
df_summary_ml_reg_iteration_1, dict_df_metrics_ml_reg_iteration_1, dict_all_targets_predictions_ml_reg_iteration_1 = run_regression_ml_tests(cv_iteration, model_names)

In [27]:
# Iteration 2: run the tests for machine learning regression models. 
cv_iteration = 2
df_summary_ml_reg_iteration_2, dict_df_metrics_ml_reg_iteration_2, dict_all_targets_predictions_ml_reg_iteration_2 = run_regression_ml_tests(cv_iteration, model_names)

In [28]:
# Iteration 3: run the tests for machine learning regression models. 
cv_iteration = 3
df_summary_ml_reg_iteration_3, dict_df_metrics_ml_reg_iteration_3, dict_all_targets_predictions_ml_reg_iteration_3 = run_regression_ml_tests(cv_iteration, model_names)

In [29]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_ml_reg_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_ml_reg_pooled = create_summary_df(model_names, 'MAE', brain_regions)

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_ml_reg_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_ml_reg_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_ml_reg_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, 'MAE', brain_regions)

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_ml_reg_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_ml_reg_pooled = store_statistics_in_summary_df(df_summary_ml_reg_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, 'MAE', brain_regions)

In [30]:
# Diplay the pooled summary DataFrame. 
df_summary_ml_reg_pooled

N mean model MAE  \
model             brain region                                             
Linear Regression Background                           27       0.840808   
                  Frontal Pole                         27       0.905702   
                  Insular Cortex                       27       0.894583   
                  Superior Frontal Gyrus               27       0.855446   
                  Middle Frontal Gyrus                 27       0.904246   
...                                                    ..            ...   
XGBoost           Planum Polare                        27       0.835909   
                  Heschl's Gyrus (includes H1 and H2)  27       0.839821   
                  Planum Temporale                     27       0.837093   
                  Supracalcarine Cortex                27       0.833106   
                  Occipital Pole                       27       0.836262   

                                                      mean baseline MAE  \
model             brain region                                            
Linear Regression Background                                   0.842854   
                  Frontal Pole                                  0.72948   
                  Insular Cortex                               0.754798   
                  Superior Frontal Gyrus                       0.719737   
                  Middle Frontal Gyrus                         0.734678   
...                                                                 ...   
XGBoost           Planum Polare                                 0.77202   
                  Heschl's Gyrus (includes H1 and H2)          0.766299   
                  Planum Temporale                             0.759918   
                  Supracalcarine Cortex                        0.799145   
                  Occipital Pole                               0.793852   

                                                      STD model MAE  \
model             brain region                                        
Linear Regression Background                                0.14465   
                  Frontal Pole                             0.165448   
                  Insular Cortex                           0.124419   
                  Superior Frontal Gyrus                   0.157173   
                  Middle Frontal Gyrus                     0.132882   
...                                                             ...   
XGBoost           Planum Polare                            0.080093   
                  Heschl's Gyrus (includes H1 and H2)      0.066278   
                  Planum Temporale                         0.066885   
                  Supracalcarine Cortex                    0.069086   
                  Occipital Pole                           0.054556   

                                                      STD baseline MAE  \
model             brain region                                           
Linear Regression Background                                  0.018467   
                  Frontal Pole                                0.126654   
                  Insular Cortex                              0.068886   
                  Superior Frontal Gyrus                      0.136084   
                  Middle Frontal Gyrus                        0.115156   
...                                                                ...   
XGBoost           Planum Polare                               0.062904   
                  Heschl's Gyrus (includes H1 and H2)         0.053353   
                  Planum Temporale                             0.04281   
                  Supracalcarine Cortex                        0.05605   
                  Occipital Pole                              0.049365   

                                                      Wilcoxon W statistic  \
model             brain region                                               
Linear Regression Background                    

## **2. Deep Learning**

We perform **Wilcoxon signed-rank tests** on the results of the deep learning models for the classification and regression tasks. 

### Classification

In [31]:
# Define a function to load the deep learning results for the classification task. 
def load_classification_dl_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'classification/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'classification/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the deep learning results for the classification task. 
    with open(iteration_result_path + 'neural_networks/dict_predictions_neural_networks_class.p', 'rb') as file:
        dict_predictions_neural_networks_class = pickle.load(file)
    with open(iteration_result_path + 'convolutional_neural_networks/dict_predictions_convolutional_neural_networks_class.p', 'rb') as file:
        dict_predictions_convolutional_neural_networks_class = pickle.load(file)
    with open(iteration_result_path + 'recurrent_neural_networks/dict_predictions_recurrent_neural_networks_class.p', 'rb') as file:
        dict_predictions_recurrent_neural_networks_class = pickle.load(file)
    with open(iteration_result_path + 'transformers/dict_predictions_transformers_class.p', 'rb') as file:
        dict_predictions_transformers_class = pickle.load(file)

    return [dict_predictions_neural_networks_class, 
            dict_predictions_convolutional_neural_networks_class, 
            dict_predictions_recurrent_neural_networks_class, 
            dict_predictions_transformers_class, 
            test_set]

In [32]:
# Define a function to run the tests for deep learning classification models.
def run_classification_dl_tests(cv_iteration, model_names):

    # Load the deep learning results for the classification task.
    dict_predictions_neural_networks_class, dict_predictions_convolutional_neural_networks_class, dict_predictions_recurrent_neural_networks_class, dict_predictions_transformers_class, test_set = load_classification_dl_results(cv_iteration)

    predictions_dl_class = [dict_predictions_neural_networks_class, 
                            dict_predictions_convolutional_neural_networks_class, 
                            dict_predictions_recurrent_neural_networks_class, 
                            dict_predictions_transformers_class]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_dl_class, dict_df_metrics_dl_class, _ = run_wilcoxon_tests(model_names, dict_targets_class, predictions_dl_class, brain_regions, 'accuracy', 'deep_learning', test_set)
    
    return df_summary_dl_class, dict_df_metrics_dl_class

In [33]:
# Define the deep learning models for classification. 
model_names = ['Multi-Layer Perceptron', 'Convolutional Neural Network', 'Recurrent Neural Network', 'Transformer']

In [34]:
# Iteration 1: run the tests for deep learning classification models. 
cv_iteration = 1
df_summary_dl_class_iteration_1, dict_df_metrics_dl_class_iteration_1 = run_classification_dl_tests(cv_iteration, model_names)

In [35]:
# Iteration 2: run the tests for deep learning classification models. 
cv_iteration = 2
df_summary_dl_class_iteration_2, dict_df_metrics_dl_class_iteration_2 = run_classification_dl_tests(cv_iteration, model_names)

In [36]:
# Iteration 3: run the tests for deep learning classification models. 
cv_iteration = 3
df_summary_dl_class_iteration_3, dict_df_metrics_dl_class_iteration_3 = run_classification_dl_tests(cv_iteration, model_names)

In [37]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_dl_class_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_dl_class_pooled = create_summary_df(model_names, 'accuracy', brain_regions)

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_dl_class_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_dl_class_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_dl_class_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, 'accuracy', brain_regions)

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_dl_class_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_dl_class_pooled = store_statistics_in_summary_df(df_summary_dl_class_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, 'accuracy', brain_regions)

In [38]:
# Diplay the pooled summary DataFrame. 
df_summary_dl_class_pooled

N  \
model                  brain region                              
Multi-Layer Perceptron Background                           27   
                       Frontal Pole                         27   
                       Insular Cortex                       27   
                       Superior Frontal Gyrus               27   
                       Middle Frontal Gyrus                 27   
...                                                         ..   
Transformer            Planum Polare                        27   
                       Heschl's Gyrus (includes H1 and H2)  27   
                       Planum Temporale                     27   
                       Supracalcarine Cortex                27   
                       Occipital Pole                       27   

                                                           mean model accuracy  \
model                  brain region                                              
Multi-Layer Perceptron Background                                     0.553837   
                       Frontal Pole                                   0.535319   
                       Insular Cortex                                 0.513746   
                       Superior Frontal Gyrus                         0.536846   
                       Middle Frontal Gyrus                           0.539901   
...                                                                        ...   
Transformer            Planum Polare                                  0.504391   
                       Heschl's Gyrus (includes H1 and H2)              0.5021   
                       Planum Temporale                               0.506109   
                       Supracalcarine Cortex                          0.507446   
                       Occipital Pole                                 0.501527   

                                                           mean baseline accuracy  \
model                  brain region                                                 
Multi-Layer Perceptron Background                                         0.41428   
                       Frontal Pole                                      0.511264   
                       Insular Cortex                                    0.522719   
                       Superior Frontal Gyrus                            0.518328   
                       Middle Frontal Gyrus                              0.512218   
...                                                                           ...   
Transformer            Planum Polare                                     0.506682   
                       Heschl's Gyrus (includes H1 and H2)               0.517373   
                       Planum Temporale                                  0.507064   
                       Supracalcarine Cortex                             0.494654   
                       Occipital Pole                                    0.507255   

                                                           STD model accuracy  \
model                  brain region                                             
Multi-Layer Perceptron Background                                    0.049655   
                       Frontal Pole                                   0.04995   
                       Insular Cortex                                0.030729   
                       Superior Frontal Gyrus                        0.045021   
                       Middle Frontal Gyrus                          0.036661   
...                                                                       ...   
Transformer            Planum Polare                                 0.026233   
                       Heschl's Gyrus (includes H1 and H2)           0.035317   
                       Planum Temporale                              0.021964   
                       Supracalcarine Cortex                         0.023979   
                       Occipital Pole                    

### Regression

In [39]:
# Define a function to load the deep learning results for the regression task. 
def load_regression_dl_results(cv_iteration):

    if cv_iteration == 1:
        iteration_result_path = results_path + 'regression/'
        test_set = 'fmriNF'
    elif cv_iteration == 2:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegfmriNF'
    elif cv_iteration == 3:
        iteration_result_path = results_path + 'regression/cross_validation/cv_iteration_' + str(cv_iteration) + '/'
        test_set = 'eegNF'

    # Load the deep learning results for the regression task. 
    with open(iteration_result_path + 'neural_networks/dict_predictions_neural_networks_reg.p', 'rb') as file:
        dict_predictions_neural_networks_reg = pickle.load(file)
    with open(iteration_result_path + 'convolutional_neural_networks/dict_predictions_convolutional_neural_networks_reg.p', 'rb') as file:
        dict_predictions_convolutional_neural_networks_reg = pickle.load(file)
    with open(iteration_result_path + 'recurrent_neural_networks/dict_predictions_recurrent_neural_networks_reg.p', 'rb') as file:
        dict_predictions_recurrent_neural_networks_reg = pickle.load(file)
    with open(iteration_result_path + 'transformers/dict_predictions_transformers_reg.p', 'rb') as file:
        dict_predictions_transformers_reg = pickle.load(file)

    return [dict_predictions_neural_networks_reg, 
            dict_predictions_convolutional_neural_networks_reg, 
            dict_predictions_recurrent_neural_networks_reg, 
            dict_predictions_transformers_reg, 
            test_set]

In [40]:
# Define a function to run the tests for deep learning regression models.
def run_regression_dl_tests(cv_iteration, model_names):

    # Load the deep learning results for the regression task.
    dict_predictions_neural_networks_reg, dict_predictions_convolutional_neural_networks_reg, dict_predictions_recurrent_neural_networks_reg, dict_predictions_transformers_reg, test_set = load_regression_dl_results(cv_iteration)

    predictions_dl_reg = [dict_predictions_neural_networks_reg, 
                            dict_predictions_convolutional_neural_networks_reg, 
                            dict_predictions_recurrent_neural_networks_reg, 
                            dict_predictions_transformers_reg]

    # Run the Wilcoxon signed-rank tests for all models. 
    df_summary_dl_reg, dict_df_metrics_dl_reg, dict_all_targets_predictions_dl_reg = run_wilcoxon_tests(model_names, dict_targets_reg, predictions_dl_reg, brain_regions, 'MAE', 'deep_learning', test_set)
    
    return df_summary_dl_reg, dict_df_metrics_dl_reg, dict_all_targets_predictions_dl_reg

In [41]:
# Define the deep learning models for regression. 
model_names = ['Multi-Layer Perceptron', 'Convolutional Neural Network', 'Recurrent Neural Network', 'Transformer']

In [42]:
# Iteration 1: run the tests for deep learning regression models. 
cv_iteration = 1
df_summary_dl_reg_iteration_1, dict_df_metrics_dl_reg_iteration_1, dict_all_targets_predictions_dl_reg_iteration_1 = run_regression_dl_tests(cv_iteration, model_names)

In [43]:
# Iteration 2: run the tests for deep learning regression models. 
cv_iteration = 2
df_summary_dl_reg_iteration_2, dict_df_metrics_dl_reg_iteration_2, dict_all_targets_predictions_dl_reg_iteration_2 = run_regression_dl_tests(cv_iteration, model_names)

In [44]:
# Iteration 3: run the tests for deep learning regression models. 
cv_iteration = 3
df_summary_dl_reg_iteration_3, dict_df_metrics_dl_reg_iteration_3, dict_all_targets_predictions_dl_reg_iteration_3 = run_regression_dl_tests(cv_iteration, model_names)

In [45]:
# Create a dictionary to store the pooled metrics DataFrames for all models. 
dict_df_metrics_dl_reg_pooled = {}

# Create a summary DataFrame to store the statistics for all models. 
df_summary_dl_reg_pooled = create_summary_df(model_names, 'MAE', brain_regions)

# Iterate through all models. 
for model_name in model_names:

    # Retrieve the model index.
    model_index = model_names.index(model_name)

    # Pool the metrics DataFrames across cross-validation iterations. 
    df_iteration_1 = dict_df_metrics_dl_reg_iteration_1[model_name]
    df_iteration_2 = dict_df_metrics_dl_reg_iteration_2[model_name]
    df_iteration_3 = dict_df_metrics_dl_reg_iteration_3[model_name]
    df_pooled = pd.concat([df_iteration_1, df_iteration_2, df_iteration_3], ignore_index = True)

    # Perform the Wilcoxon signed-rank test. 
    statistics_scipy, statistics_pingouin = wilcoxon_test(df_pooled, 'MAE', brain_regions)

    # Store the pooled metrics DataFrame in the dictionary. 
    dict_df_metrics_dl_reg_pooled[model_name] = df_pooled

    # Store the statistics in the summary DataFrame. 
    df_summary_dl_reg_pooled = store_statistics_in_summary_df(df_summary_dl_reg_pooled, model_names, model_index, statistics_scipy, statistics_pingouin, 'MAE', brain_regions)

In [46]:
# Diplay the pooled summary DataFrame. 
df_summary_dl_reg_pooled

N mean model MAE  \
model                  brain region                                             
Multi-Layer Perceptron Background                           27       0.845659   
                       Frontal Pole                         27       0.728018   
                       Insular Cortex                       27       0.765594   
                       Superior Frontal Gyrus               27       0.733013   
                       Middle Frontal Gyrus                 27       0.735631   
...                                                         ..            ...   
Transformer            Planum Polare                        27       0.784712   
                       Heschl's Gyrus (includes H1 and H2)  27       0.784687   
                       Planum Temporale                     27        0.76632   
                       Supracalcarine Cortex                27       0.799185   
                       Occipital Pole                       27       0.785974   

                                                           mean baseline MAE  \
model                  brain region                                            
Multi-Layer Perceptron Background                                   0.842854   
                       Frontal Pole                                  0.72948   
                       Insular Cortex                               0.754798   
                       Superior Frontal Gyrus                       0.719737   
                       Middle Frontal Gyrus                         0.734678   
...                                                                      ...   
Transformer            Planum Polare                                 0.77202   
                       Heschl's Gyrus (includes H1 and H2)          0.766299   
                       Planum Temporale                             0.759918   
                       Supracalcarine Cortex                        0.799145   
                       Occipital Pole                               0.793852   

                                                           STD model MAE  \
model                  brain region                                        
Multi-Layer Perceptron Background                               0.027404   
                       Frontal Pole                             0.126874   
                       Insular Cortex                           0.072604   
                       Superior Frontal Gyrus                   0.143832   
                       Middle Frontal Gyrus                     0.115498   
...                                                                  ...   
Transformer            Planum Polare                            0.072543   
                       Heschl's Gyrus (includes H1 and H2)      0.065722   
                       Planum Temporale                         0.040772   
                       Supracalcarine Cortex                    0.073807   
                       Occipital Pole                           0.052904   

                                                           STD baseline MAE  \
model                  brain region                                           
Multi-Layer Perceptron Background                                  0.018467   
                       Frontal Pole                                0.126654   
                       Insular Cortex                              0.068886   
                       Superior Frontal Gyrus                      0.136084   
                       Middle Frontal Gyrus                        0.115156   
...                                                                     ...   
Transformer            Planum Polare                               0.062904   
                       Heschl's Gyrus (includes H1 and H2)         0.053353   
                       Planum Temporale                             0.04281   
                       Supracalcarine Cortex                        0.05605   
                       Occipi

## **3. Foundation Models**

In [47]:
# Load the foundation models results. 
with open(results_path + 'classification/dict_predictions_gemma.p', 'rb') as file:
    dict_predictions_gemma = pickle.load(file)
with open(results_path + 'classification/dict_predictions_llama.p', 'rb') as file:
    dict_predictions_llama = pickle.load(file)
with open(results_path + 'classification/dict_predictions_gemma_CoT.p', 'rb') as file:
    dict_predictions_gemma_CoT = pickle.load(file)
with open(results_path + 'classification/dict_predictions_gemma_no_finetuning.p', 'rb') as file:
    dict_predictions_gemma_no_finetuning = pickle.load(file)
with open(results_path + 'classification/dict_predictions_gemma_finetuning.p', 'rb') as file:
    dict_predictions_gemma_finetuning = pickle.load(file)
with open(results_path + 'classification/dict_predictions_paligemma.p', 'rb') as file:
    dict_predictions_paligemma = pickle.load(file)
with open(results_path + 'multi-channel/dict_predictions_gemma_5_channels.p', 'rb') as file:
    dict_predictions_gemma_5_channels = pickle.load(file)

# Load the brain regions. 
with open(results_path + 'brain_regions_large_language_models.p', 'rb') as file:
    dict_brain_regions_large_language_models = pickle.load(file)
with open(results_path + 'brain_regions_large_language_model_CoT.p', 'rb') as file:
    dict_brain_regions_large_language_model_CoT = pickle.load(file)
with open(results_path + 'brain_regions_large_language_model_fine-tuning.p', 'rb') as file:
    dict_brain_regions_large_language_model_fine_tuning = pickle.load(file)
with open(results_path + 'brain_regions_large_multimodal_model.p', 'rb') as file:
    dict_brain_regions_large_multimodal_model = pickle.load(file)
with open(results_path + 'multi-channel/brain_regions_multi-channel_large_language_models.p', 'rb') as file:
    dict_brain_regions_multi_channel_large_language_models = pickle.load(file)

# Load the fMRI scans. 
with open(results_path + 'fmri_scans_large_language_models.p', 'rb') as file:
    dict_fmri_scans_large_language_models = pickle.load(file)
with open(results_path + 'fmri_scans_large_language_model_CoT.p', 'rb') as file:
    dict_fmri_scans_large_language_model_CoT = pickle.load(file)
with open(results_path + 'fmri_scans_large_language_model_fine-tuning.p', 'rb') as file:
    dict_fmri_scans_large_language_model_fine_tuning = pickle.load(file)
with open(results_path + 'fmri_scans_large_multimodal_model.p', 'rb') as file:
    dict_fmri_scans_large_multimodal_model = pickle.load(file)
with open(results_path + 'multi-channel/fmri_scans_multi-channel_large_language_models.p', 'rb') as file:
    dict_fmri_scans_multi_channel_large_language_models = pickle.load(file)

In [48]:
# Define the foundation models. 
model_names = ['Gemma', 'Llama', 'Gemma Chain-of-Thought', 'Gemma Without Fine-Tuning', 'Gemma With Fine-Tuning', 'PaliGemma', 'Gemma 5-Channel']
list_dict_predictions = [dict_predictions_gemma, 
                         dict_predictions_llama, 
                         dict_predictions_gemma_CoT, 
                         dict_predictions_gemma_no_finetuning, 
                         dict_predictions_gemma_finetuning, 
                         dict_predictions_paligemma, 
                         dict_predictions_gemma_5_channels]

# Define the brain regions. 
list_brain_regions = [dict_brain_regions_large_language_models['Large Language Models: Gemma'],
                      dict_brain_regions_large_language_models['Large Language Models: Llama'],
                      dict_brain_regions_large_language_model_CoT['Large Language Model Chain-of-Thought'],
                      dict_brain_regions_large_language_model_fine_tuning['Large Language Model Fine-Tuning: Without Fine-Tuning'],
                      dict_brain_regions_large_language_model_fine_tuning['Large Language Model Fine-Tuning: With Fine-Tuning'], 
                      dict_brain_regions_large_multimodal_model['Large Multimodal Model'],
                      dict_brain_regions_multi_channel_large_language_models['Large Language Models: Gemma 5 Channels']]

# Define the fMRI scans indexes. 
list_fmri_scans_indexes = [dict_fmri_scans_large_language_models['Large Language Models: Gemma'],
                           dict_fmri_scans_large_language_models['Large Language Models: Llama'],
                           dict_fmri_scans_large_language_model_CoT['Large Language Model Chain-of-Thought'],
                           dict_fmri_scans_large_language_model_fine_tuning['Large Language Model Fine-Tuning: Without Fine-Tuning'],
                           dict_fmri_scans_large_language_model_fine_tuning['Large Language Model Fine-Tuning: With Fine-Tuning'], 
                           dict_fmri_scans_large_multimodal_model['Large Multimodal Model'], 
                           dict_fmri_scans_multi_channel_large_language_models['Large Language Models: Gemma 5 Channels']]

In [49]:
# Define a function to create the DataFrame for the statistical tests. 
def create_df_foundation_models(dict_targets, dict_predictions, brain_region_indexes, fmri_scans_indexes):
    
    # Define a Pandas DataFrames and two lists to store the results. 
    df = pd.DataFrame(columns = ['subject', 'brain_region', 'model_accuracy', 'baseline_accuracy'])
    list_targets = []
    list_predictions = []
    counter = 0

    # Iterate through all subjects. 
    for subject in subjects:

        # Iterate through all brain regions. 
        for brain_region_index in brain_region_indexes:
            
            # Fill the DataFrame with the subject and brain region. 
            df.loc[counter, 'subject'] = subject
            df.loc[counter, 'brain_region'] = brain_regions[brain_region_index] # Here, brain_regions is the global variable corresponding to all 49 regions. 

            # Retrieve the targets and predictions values for the current subject and brain region. 
            targets_values = dict_targets[subject]['eegfmriNF'][fmri_scans_indexes, brain_region_index]
            predictions_values = dict_predictions[brain_regions[brain_region_index]][subject] # Here, brain_regions is the global variable corresponding to all 49 regions. 

            # Add the targets and predictions to the lists. 
            list_targets.append(targets_values)
            list_predictions.append(predictions_values)

            # Remove the cases where the prediction is -1. 
            selected_predictions = predictions_values != -1
            targets_values = targets_values[selected_predictions]
            predictions_values = predictions_values[selected_predictions]

            # Compute the accuracy and fill in the DataFrame with the model score. 
            model_score = np.mean(targets_values == predictions_values)
            df.loc[counter, 'model_accuracy'] = model_score

            # Define the values and compute the targets distribution. 
            values = [0, 1]
            targets_distribution = [np.sum(targets_values == 0) / len(targets_values), 
                                    np.sum(targets_values == 1) / len(targets_values)]

            # Define the number of iterations. 
            iterations = 1000
            baseline_score = 0

            # Compute the baseline score using random sampling. 
            for i in range(iterations):
                samples = np.random.choice(values, size = len(targets_values), p = targets_distribution)
                baseline_score += np.mean(targets_values == samples)

            # Compute the mean baseline score. 
            baseline_score /= iterations

            # Fill in the DataFrame with the baseline score. 
            df.loc[counter, 'baseline_accuracy'] = baseline_score

            # Increment. 
            counter += 1

    return df, list_targets, list_predictions

In [50]:
# Create a function to retrieve foundation model elements. 
def retrieve_foundation_model_elements(model_index, list_dict_predictions, list_brain_regions, list_fmri_scans_indexes):

    # Retrieve the foundation model elements. 
    dict_predictions = list_dict_predictions[model_index]
    brain_region_indexes = [brain_regions.index(region) for region in list_brain_regions[model_index]] # Here, brain_regions is the global variable corresponding to all 49 regions. 
    fmri_scans_indexes = list_fmri_scans_indexes[model_index]

    return dict_predictions, brain_region_indexes, fmri_scans_indexes

In [51]:
# Example: Create and display the DataFrame for the Gemma model. 
model_index = 0
dict_predictions, brain_region_indexes, fmri_scans_indexes = retrieve_foundation_model_elements(model_index, 
                                                                                                list_dict_predictions,
                                                                                                list_brain_regions,
                                                                                                list_fmri_scans_indexes)

# Create and display the DataFrame. 
df_gemma, _, _ = create_df_foundation_models(dict_targets_foundation_models, 
                                             dict_predictions, 
                                             brain_region_indexes, 
                                             fmri_scans_indexes)
df_gemma

,subject,brain_region,model_accuracy,baseline_accuracy
0,sub-xp101,Frontal Pole,0.35,0.50555
1,sub-xp101,Insular Cortex,0.65,0.50665
2,sub-xp101,"Inferior Frontal Gyrus, pars triangularis",0.55,0.50885
3,sub-xp101,Precentral Gyrus,0.529412,0.583941
4,sub-xp101,"Superior Temporal Gyrus, posterior division",0.4,0.5008
...,...,...,...,...
67,sub-xp110,Precentral Gyrus,0.45,0.52415
68,sub-xp110,"Superior Temporal Gyrus, posterior division",0.45,0.50355
69,sub-xp110,Superior Parietal Lobule,0.6,0.5056
70,sub-xp110,Angular Gyrus,0.210526,0.513158


In [52]:
# Perform Wilcoxon signed-rank test. 
statistics_scipy_gemma, statistics_pingouin_gemma = wilcoxon_test(df_gemma, 'accuracy', brain_regions = df_gemma['brain_region'].unique())

In [53]:
# Display the SciPy statistics. 
statistics_scipy_gemma

,brain_region,N,mean_model_accuracy,mean_baseline_accuracy,std_model_accuracy,std_baseline_accuracy,wilcoxon_stat,p_value,ci_low,ci_high
0,Frontal Pole,9,0.547739,0.506676,0.113851,0.005492,33.0,0.125000,0.476979,0.616695
1,Insular Cortex,9,0.516667,0.519978,0.099084,0.036330,22.0,0.544922,0.454548,0.576902
2,"Inferior Frontal Gyrus, pars triangularis",9,0.504971,0.529738,0.125142,0.043106,20.0,0.632812,0.428363,0.580702
3,Precentral Gyrus,9,0.517068,0.539728,0.113596,0.043439,17.0,0.751953,0.445894,0.585091
4,"Superior Temporal Gyrus, posterior division",9,0.520825,0.532659,0.072325,0.035742,23.0,0.500000,0.475763,0.563548
5,Superior Parietal Lobule,9,0.556725,0.541408,0.066857,0.057345,28.0,0.285156,0.513093,0.595906
6,Angular Gyrus,9,0.502989,0.520001,0.131990,0.021964,24.0,0.455078,0.388304,0.565238
7,Occipital Pole,9,0.606140,0.540448,0.106992,0.027748,36.0,0.064453,0.533918,0.665789


In [54]:
# Display the Pingouin statistics. 
statistics_pingouin_gemma

,brain_region,W-val,alternative,p-val,RBC,CLES
0,Frontal Pole,33.0,greater,0.125000,0.466667,0.666667
1,Insular Cortex,22.0,greater,0.544922,-0.022222,0.506173
2,"Inferior Frontal Gyrus, pars triangularis",20.0,greater,0.632812,-0.111111,0.407407
3,Precentral Gyrus,17.0,greater,0.751953,-0.244444,0.456790
4,"Superior Temporal Gyrus, posterior division",23.0,greater,0.500000,0.022222,0.518519
5,Superior Parietal Lobule,28.0,greater,0.285156,0.244444,0.580247
6,Angular Gyrus,24.0,greater,0.455078,0.066667,0.580247
7,Occipital Pole,36.0,greater,0.064453,0.600000,0.765432


In [55]:
# Define a function to perform a Wilcoxon signed-rank test on a series of models. 
def run_wilcoxon_tests_foundation_models(model_names, dict_targets, list_dict_predictions, list_brain_regions, list_fmri_scans_indexes):

    # Define the brain regions, assuming they are the same for all models. 
    brain_regions_foundation_models = list_brain_regions[0]

    # Create a summary DataFrame to store the statistics for all models. 
    df_summary = create_summary_df(model_names, 'accuracy', brain_regions_foundation_models)

    # Iterate through all models. 
    for model_index in range(len(model_names)):

        # Retrieve the predictions, brain region indexes, and fMRI scans indexes. 
        dict_predictions, brain_region_indexes, fmri_scans_indexes = retrieve_foundation_model_elements(model_index, 
                                                                                                        list_dict_predictions,
                                                                                                        list_brain_regions,
                                                                                                        list_fmri_scans_indexes)

        # Create the DataFrame and perform the Wilcoxon signed-rank test. 
        df, _, _ = create_df_foundation_models(dict_targets, dict_predictions, brain_region_indexes, fmri_scans_indexes)
        statistics_scipy, statistics_pingouin = wilcoxon_test(df, 'accuracy', brain_regions_foundation_models)

        # Store the statistics in the summary DataFrame. 
        df_summary = store_statistics_in_summary_df(df_summary, model_names, model_index, statistics_scipy, statistics_pingouin, 'accuracy', brain_regions_foundation_models)
        
    # Return the summary DataFrame. 
    return df_summary

In [56]:
# Identify the models with 8 regions. 
models_with_8_regions = ['Gemma', 'Llama', 'PaliGemma', 'Gemma 5-Channel']
indexes_models_with_8_regions = [model_names.index(model_name) for model_name in models_with_8_regions]

# Select the predictions, brain regions, and fMRI scans for the models with 8 regions. 
list_dict_predictions_8_regions = [list_dict_predictions[i] for i in indexes_models_with_8_regions]
list_brain_regions_8_regions = [list_brain_regions[i] for i in indexes_models_with_8_regions]
list_fmri_scans_indexes_8_regions = [list_fmri_scans_indexes[i] for i in indexes_models_with_8_regions]

# Run the Wilcoxon signed-rank tests for models with 8 regions. 
df_summary_foundation_models_8_regions = run_wilcoxon_tests_foundation_models(models_with_8_regions, 
                                                                              dict_targets_foundation_models, 
                                                                              list_dict_predictions_8_regions, 
                                                                              list_brain_regions_8_regions,
                                                                              list_fmri_scans_indexes_8_regions)
df_summary_foundation_models_8_regions

N  \
model           brain region                                     
Gemma           Frontal Pole                                 9   
                Insular Cortex                               9   
                Inferior Frontal Gyrus, pars triangularis    9   
                Precentral Gyrus                             9   
                Superior Temporal Gyrus, posterior division  9   
                Superior Parietal Lobule                     9   
                Angular Gyrus                                9   
                Occipital Pole                               9   
Llama           Frontal Pole                                 9   
                Insular Cortex                               9   
                Inferior Frontal Gyrus, pars triangularis    9   
                Precentral Gyrus                             9   
                Superior Temporal Gyrus, posterior division  9   
                Superior Parietal Lobule                     9   
                Angular Gyrus                                9   
                Occipital Pole                               9   
PaliGemma       Frontal Pole                                 9   
                Insular Cortex                               9   
                Inferior Frontal Gyrus, pars triangularis    9   
                Precentral Gyrus                             9   
                Superior Temporal Gyrus, posterior division  9   
                Superior Parietal Lobule                     9   
                Angular Gyrus                                9   
                Occipital Pole                               9   
Gemma 5-Channel Frontal Pole                                 9   
                Insular Cortex                               9   
                Inferior Frontal Gyrus, pars triangularis    9   
                Precentral Gyrus                             9   
                Superior Temporal Gyrus, posterior division  9   
                Superior Parietal Lobule                     9   
                Angular Gyrus                                9   
                Occipital Pole                               9   

                                                            mean model accuracy  \
model           brain region                                                      
Gemma           Frontal Pole                                           0.547739   
                Insular Cortex                                         0.516667   
                Inferior Frontal Gyrus, pars triangularis              0.504971   
                Precentral Gyrus                                       0.517068   
                Superior Temporal Gyrus, posterior division            0.520825   
                Superior Parietal Lobule                               0.556725   
                Angular Gyrus                                          0.502989   
                Occipital Pole                                          0.60614   
Llama           Frontal Pole                                           0.507276   
                Insular Cortex                                         0.472986   
                Inferior Frontal Gyrus, pars triangularis              0.534698   
                Precentral Gyrus                                       0.500677   
                Superior Temporal Gyrus, posterior division            0.461763   
                Superior Parietal Lobule                               0.528772   
                Angular Gyrus                                          0.543137   
                Occipital Pole                                         0.528162   
PaliGemma       Frontal Pole                                           0.427513   
                Insular Cortex                                         0.529938   
                Inferior Frontal Gyrus, pars triangularis               0.48858   
                Precentral Gyrus                                       0.

In [57]:
# Identify the models with 5 regions. 
models_with_5_regions = ['Gemma Chain-of-Thought', 'Gemma Without Fine-Tuning', 'Gemma With Fine-Tuning']
indexes_models_with_5_regions = [model_names.index(model_name) for model_name in models_with_5_regions]

# Select the predictions, brain regions, and fMRI scans for the models with 5 regions. 
list_dict_predictions_5_regions = [list_dict_predictions[i] for i in indexes_models_with_5_regions]
list_brain_regions_5_regions = [list_brain_regions[i] for i in indexes_models_with_5_regions]
list_fmri_scans_indexes_5_regions = [list_fmri_scans_indexes[i] for i in indexes_models_with_5_regions]

# Run the Wilcoxon signed-rank tests for models with 5 regions. 
df_summary_foundation_models_5_regions = run_wilcoxon_tests_foundation_models(models_with_5_regions, 
                                                                              dict_targets_foundation_models, 
                                                                              list_dict_predictions_5_regions, 
                                                                              list_brain_regions_5_regions,
                                                                              list_fmri_scans_indexes_5_regions)
df_summary_foundation_models_5_regions

N  \
model                     brain region                                     
Gemma Chain-of-Thought    Frontal Pole                                 9   
                          Precentral Gyrus                             9   
                          Superior Temporal Gyrus, posterior division  9   
                          Superior Parietal Lobule                     9   
                          Occipital Pole                               9   
Gemma Without Fine-Tuning Frontal Pole                                 9   
                          Precentral Gyrus                             9   
                          Superior Temporal Gyrus, posterior division  9   
                          Superior Parietal Lobule                     9   
                          Occipital Pole                               9   
Gemma With Fine-Tuning    Frontal Pole                                 9   
                          Precentral Gyrus                             9   
                          Superior Temporal Gyrus, posterior division  9   
                          Superior Parietal Lobule                     9   
                          Occipital Pole                               9   

                                                                      mean model accuracy  \
model                     brain region                                                      
Gemma Chain-of-Thought    Frontal Pole                                           0.495108   
                          Precentral Gyrus                                       0.597953   
                          Superior Temporal Gyrus, posterior division            0.527959   
                          Superior Parietal Lobule                               0.539409   
                          Occipital Pole                                         0.562736   
Gemma Without Fine-Tuning Frontal Pole                                           0.529654   
                          Precentral Gyrus                                        0.57076   
                          Superior Temporal Gyrus, posterior division            0.525276   
                          Superior Parietal Lobule                                0.55679   
                          Occipital Pole                                         0.542982   
Gemma With Fine-Tuning    Frontal Pole                                           0.471637   
                          Precentral Gyrus                                       0.594899   
                          Superior Temporal Gyrus, posterior division            0.528687   
                          Superior Parietal Lobule                                0.58681   
                          Occipital Pole                                          0.55369   

                                                                      mean baseline accuracy  \
model                     brain region                                                         
Gemma Chain-of-Thought    Frontal Pole                                              0.506453   
                          Precentral Gyrus                                          0.535631   
                          Superior Temporal Gyrus, posterior division               0.529547   
                          Superior Parietal Lobule                                  0.538205   
                          Occipital Pole                                            0.545215   
Gemma Without Fine-Tuning Frontal Pole                                              0.507328   
                          Precentral Gyrus                                          0.542402   
                          Superior Temporal Gyrus, posterior division               0.529853   
                          Superior Parietal Lobule                                  0.539211   
                          Occipital Pole                                            0.543706   
Gemma With Fine-Tuning    Frontal Pole      

## **4. Results**

In [58]:
# If the cross-validation results path does not exist, create it. 
if not os.path.exists(results_path + 'regions_of_interest/'):
    os.makedirs(results_path + 'regions_of_interest/')
    
# Store the general results in a dictionary. 
general_results = dict()
general_results['df_summary_ml_class_pooled'] = df_summary_ml_class_pooled
general_results['df_summary_ml_reg_pooled'] = df_summary_ml_reg_pooled
general_results['df_summary_dl_class_pooled'] = df_summary_dl_class_pooled
general_results['df_summary_dl_reg_pooled'] = df_summary_dl_reg_pooled
general_results['df_summary_foundation_models_8_regions'] = df_summary_foundation_models_8_regions
general_results['df_summary_foundation_models_5_regions'] = df_summary_foundation_models_5_regions

# Save the general results into a Pickle file. 
with open(results_path + 'regions_of_interest/general_results.p', 'wb') as file:
    pickle.dump(general_results, file)

In [59]:
# Display the keys of the general results dictionary. 
list(general_results.keys())

['df_summary_ml_class_pooled',
 'df_summary_ml_reg_pooled',
 'df_summary_dl_class_pooled',
 'df_summary_dl_reg_pooled',
 'df_summary_foundation_models_8_regions',
 'df_summary_foundation_models_5_regions']